## Introduction
Transportation access plays a major role in commute times, quality of life, and regional equity. This project analyzes Bay Area commute data to evaluate how travel time differs by transportation mode, including driving alone, carpooling, and public transit.

By comparing commute times across cities, this analysis identifies where transit systems are competitive with driving and where commuters face longer travel times when relying on public transportation. These differences help illustrate how infrastructure and urban form influence mobility and overall commute burden.

Cities with missing public transit commute times were removed from the analysis, as they cannot contribute to insights about transit efficiency. This left 60 cities for transit related comparisons.

## Objectives
• Compare average commute times across transportation modes (driving alone, carpooling, and public transit)

• Evaluate how transit commute times differ between cities and how competitive transit is relative to driving

• Identify cities with strong transit accessibility based on travel time efficiency

• Highlight cities where transit commutes are significantly longer, indicating potential transportation gaps

• Examine how overall commute burden relates to the availability of efficient public transportation

## Commute Times Dataset
The dataset used in this analysis comes from the Bay Area Metro Vital Signs data portal, which provides regional indicators related to transportation, housing, and economic conditions. The specific dataset contains average commute times for Bay Area cities, broken down by transportation mode.

Each row represents a city-year combination and includes the following key columns:

jurisdiction – City or municipality where commuters reside

county – County associated with the city

year – Year the data was recorded

overall – Average commute time (minutes) across all transportation modes

drive_alone – Average commute time for commuters driving alone

carpool – Average commute time for commuters who carpool

transit – Average commute time for commuters using public transportation

walking – Average commute time for commuters who walk

source – Data source reference provided in the dataset

geometry – Geographic boundary data used for mapping and spatial analysis

Commute times are reported in minutes, allowing for direct comparison of travel burden across cities and transportation methods.

### Building the dataset

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# Load dataset and verify
transitdf = pd.read_csv('/workspaces/bay-area-commute-analysis/data/raw/Vital_Signs__Commute_Time_-_Cities_20260128.csv')
print(transitdf.head())

  jurisdiction   county   year    overall  drive_alone    carpool    transit  \
0      Alameda  Alameda  2,023  32.058017    28.568969  31.174674  49.872232   
1       Albany  Alameda  2,023  32.654894    28.664629  22.493036  50.307927   
2     Berkeley  Alameda  2,023  29.909062    28.897937  25.424362  48.796393   
3       Dublin  Alameda  2,023  37.806120    34.477289  35.424588  70.035601   
4   Emeryville  Alameda  2,023  30.503385    27.952522  29.770916  48.000000   

        walk                 source  \
0  23.343496  ACS_B08136_B08301_5YR   
1  13.220165  ACS_B08136_B08301_5YR   
2  15.245429  ACS_B08136_B08301_5YR   
3   8.469203  ACS_B08136_B08301_5YR   
4  18.687050  ACS_B08136_B08301_5YR   

                                            geometry  
0  MULTIPOLYGON (((-122.28286719299996 37.7637559...  
1  MULTIPOLYGON (((-122.28818215799998 37.8978400...  
2  MULTIPOLYGON (((-122.24692314999999 37.8853600...  
3  MULTIPOLYGON (((-121.85505706899994 37.7014150...  
4  MULTIP

### Data inspection

In [7]:
#Check column info, inpsect data types, check for missing values.
print(transitdf.dtypes)
#Commute times are numeric, however year is string datatype, will have to be converted to int. 
print('Null Data? ' + str(transitdf.isnull().values.any()))
#Missing data confirmed in some columns.
print('Same year? ' + str(transitdf['year'].nunique()==1))
#All data is confirmed from the same year.


jurisdiction        str
county              str
year                str
overall         float64
drive_alone     float64
carpool         float64
transit         float64
walk            float64
source              str
geometry            str
dtype: object
Null Data? True
Same year? True


### Data cleaning and preperation

In [8]:
#Convert year to int, not really important in this instance but if the dataset had different years, it would allow us to filter properly.
transitdf['year']=transitdf['year'].str.replace(',','').astype(int)
print(transitdf.info())
#41 cities in the dataset have no data for transit commute times, and will be removed.
transitdf=transitdf.dropna(subset=['transit'])


<class 'pandas.DataFrame'>
RangeIndex: 101 entries, 0 to 100
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   jurisdiction  101 non-null    str    
 1   county        101 non-null    str    
 2   year          101 non-null    int64  
 3   overall       101 non-null    float64
 4   drive_alone   63 non-null     float64
 5   carpool       62 non-null     float64
 6   transit       60 non-null     float64
 7   walk          63 non-null     float64
 8   source        101 non-null    str    
 9   geometry      101 non-null    str    
dtypes: float64(5), int64(1), str(4)
memory usage: 8.0 KB
None


### Feature engineering/Analysis prep

In [ ]:
# Transit penalty: extra minutes vs driving alone
transitdf['transit_penalty'] = transitdf['transit'] - transitdf['drive_alone']

# Carpool penalty: extra minutes vs driving alone
transitdf['carpool_penalty'] = transitdf['carpool'] - transitdf['drive_alone']
# Average penalty: Average time added to your commute by choosing not to drive
print(transitdf[['jurisdiction','transit_penalty']].sort_values(by='transit_penalty',ascending=True))

           jurisdiction  transit_penalty
23        San Francisco        13.715542
79             Pacifica        16.120673
82            San Bruno        17.059612
71                Colma        18.475655
12               Orinda        18.824014
27              Oakland        18.824296
2              Berkeley        19.898456
81         Redwood City        19.919557
4            Emeryville        20.047478
36               Moraga        20.996600
0               Alameda        21.303262
72            Daly City        21.365442
33           El Cerrito        21.419903
1                Albany        21.643298
8           San Leandro        21.912254
58        Mountain View        21.980272
34            Lafayette        22.576794
69           Burlingame        23.220246
66            Sunnyvale        23.574120
24             Atherton        23.605668
85  South San Francisco        23.676779
84            San Mateo        24.267292
78             Millbrae        25.027687
28             P

## Bart Ridership Dataset

The dataset in this analysis comes directly from the bart.gov website. The dataset contains average weekday ridership for each station in the Bay Area Rapid Transit system for January 2026.

Each entry represents a single station and includes the average number of riders entering or exiting that station on a typical weekday. The dataset also includes monthly and yearly ridership changes compared to previous periods. 

For the purpose of this analysis, the primary data we are interested in is the average weekday ridership for each station. This metric will be used to indentify stations with low ridership and explore whether those stations are located in areas with weaker public transit accessibility. 

It's important to note that the stations are not grouped by juristiction or city. Each station will have to be manually associated with their respective juristiction later on in the analysis so that ridership data can be properly compared with city level commute data.

### Building the dataset

In [ ]:
ridershipdf=pd.read_csv('/workspaces/bay-area-commute-analysis/data/raw/bart_ridership_jan2026.csv')
ridershipdf.head()
ridershipdf.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   station             50 non-null     str  
 1   avg_weekday_riders  50 non-null     str  
 2   monthly_change      50 non-null     str  
 3   monthly_percent     50 non-null     str  
 4   yearly_change       50 non-null     str  
 5   yearly_percent      50 non-null     str  
dtypes: str(6)
memory usage: 2.9 KB


,station,avg_weekday_riders,monthly_change,monthly_percent,yearly_change,yearly_percent
count,50,50,50,50,50,50
unique,50,50,47,39,47,45
top,Richmond,"2,470",+175,+9.3%,+36,-5.6%
freq,1,1,3,3,2,2


### Data Inspection

In [ ]:
print(ridershipdf.dtypes)
#Ridership count and percentage change is stored as string, should be converted to int.
print('Null data? ' + str(ridershipdf.isnull().values.any()))
#Missing data in some columns.
print(ridershipdf.shape)
#60 rows despite only 50 existing bart stations. Should be rechecked after null values are removed.
print(ridershipdf.columns)
#Column names are correct
print('Duplicate rows:', ridershipdf.duplicated().sum())
#Nine duplicate rows.

station               str
avg_weekday_riders    str
monthly_change        str
monthly_percent       str
yearly_change         str
yearly_percent        str
dtype: object
Null data? True
(60, 6)
Index(['station', 'avg_weekday_riders', 'monthly_change', 'monthly_percent',
       'yearly_change', 'yearly_percent'],
      dtype='str')
Duplicate rows: 9
